# European Cross-Commodity Risk Monitor v12
## Gas + Carbon → Power Curve Implications

**Author:** Clifford Pang (cliff4dp)  
**Version:** 12.0 | May 2026


---
## 1. Setup

In [ ]:
# ==================== 1. Setup ====================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yfinance as yf
import requests
import json
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')

print("v11 Notebook - Improved Headers + Comments")

---
## 2. Interpretation Rules

In [ ]:
# ==================== 2. Interpretation Rules ====================
RULES = {
    "TTF": {
        "tight": "> 55 EUR/MWh → Strong tightness, bullish for power curve",
        "moderate": "45–55 EUR/MWh → Moderate tightness (current regime)",
        "loose": "< 40 EUR/MWh → Oversupply, bearish for power"
    },
    "EUA": {
        "strong": "> 75 EUR/t → Strong policy floor",
        "constructive": "65–75 EUR/t → Constructive (current range)",
        "weak": "< 60 EUR/t → Weak policy signal"
    },
    "Spark_Spread": {
        "positive": "> 0 → Gas plants profitable",
        "negative": "< 0 → Gas plants unprofitable → Bullish for power"
    }
}

STORAGE_RULES = {
    "critical": "< -10pp → Critically low, strongly bullish",
    "tight": "-10pp to -5pp → Tight, bullish",
    "normal": "-5pp to +5pp → Normal",
    "comfortable": "> +5pp → Comfortable"
}

print("Rules loaded.")

---
## 3. Data Collection

### 3.1a TTF Price + 30-Day Stats (Automated)

In [ ]:
# ==================== 3.1 TTF Price + 30-Day Stats (Automated) ====================
# Pulls the last 30 days of TTF futures prices for current price and 30-day statistics
def get_ttf_analysis():
    try:
        ttf = yf.Ticker("TTF=F")
        hist = ttf.history(period="30d")

        current = round(hist['Close'].iloc[-1], 2)
        change_30d = round(((current - hist['Close'].iloc[0]) / hist['Close'].iloc[0]) * 100, 1)
        high_30d = round(hist['Close'].max(), 2)
        low_30d = round(hist['Close'].min(), 2)
        avg_30d = round(hist['Close'].mean(), 2)

        return {
            "current": current,
            "change_30d": change_30d,
            "high_30d": high_30d,
            "low_30d": low_30d,
            "avg_30d": avg_30d
        }
    except:
        return {"current": 48.5, "change_30d": 3.8, "high_30d": 52.1, "low_30d": 44.2, "avg_30d": 47.8}

ttf_data = get_ttf_analysis()
TTF_PRICE = ttf_data["current"]

print(f"[3.1] TTF Analysis (30 days):")
print(f"      Current: €{TTF_PRICE} | 30d Change: {ttf_data['change_30d']}%")
print(f"      30d High: €{ttf_data['high_30d']} | Low: €{ttf_data['low_30d']} | Avg: €{ttf_data['avg_30d']}")

### 3.1b TTF 6-Month Historical Data (For Charts)

In [ ]:
# ==================== 3.1b TTF 6-Month Historical Data (For Charts) ====================
# Loads 6 months of TTF data for Chart 1 (TTF vs Storage) and Chart 4 (Volatility)
def get_ttf_6m_history():
    try:
        ttf = yf.Ticker("TTF=F")
        hist = ttf.history(period="6mo")
        return hist['Close'].round(2)
    except:
        return None

ttf_6m = get_ttf_6m_history()
print(f"[3.1b] TTF 6-Month History loaded: {len(ttf_6m)} days" if ttf_6m is not None else "Failed to load 6m TTF data")

### 3.1c Historical Realized Volatility (6 months)

In [ ]:
# ==================== 3.1c Historical Realized Volatility (6 months) ====================
# Calculates rolling 30-day realized volatility from the 6-month TTF data loaded in 3.1b
# Resamples to weekly frequency for Chart 4 (TTF Price vs Realized Volatility)
def get_historical_volatility():
    try:
        returns = np.log(ttf_6m / ttf_6m.shift(1)).dropna()
        rolling_vol = returns.rolling(window=30).std() * np.sqrt(252) * 100
        weekly_vol = rolling_vol.resample('W').mean().round(1)

        print(f"[3.1c] Historical volatility calculated: {len(weekly_vol)} weeks")
        return weekly_vol
    except:
        return None

historical_vol = get_historical_volatility()

### 3.2 TTF 30-Day Realized Volatility (Automated)

In [ ]:
# ==================== 3.2 TTF 30-Day Realized Volatility (Automated) ====================
# Calculates annualized 30-day realized volatility using daily log returns
def get_ttf_volatility():
    try:
        ttf = yf.Ticker("TTF=F")
        hist = ttf.history(period="30d")

        returns = np.log(hist['Close'] / hist['Close'].shift(1)).dropna()
        vol = np.std(returns) * np.sqrt(252) * 100
        return round(vol, 1)
    except:
        return 42.5

TTF_VOL = get_ttf_volatility()
print(f"[3.2] TTF 30-Day Realized Volatility = {TTF_VOL}%")

### 3.3a EU Gas Storage Historical Pull (Run once per month)

In [ ]:
# ==================== 3.3 EU Gas Storage Historical Pull (Run once per month) ====================
# Fetches 5 years of historical EU storage data and saves monthly statistics to JSON
API_KEY = "ae22b60e0943c9239aa6027a51a140b1"

def fetch_historical_storage(years=5):
    from datetime import datetime, timedelta

    end_date = datetime.now().strftime('%Y-%m-%d')
    start_date = (datetime.now() - timedelta(days=years*365)).strftime('%Y-%m-%d')

    url = f"https://agsi.gie.eu/api?type=eu&from={start_date}&to={end_date}&size=300"
    headers = {"x-key": API_KEY}

    all_data = []
    page = 1

    while True:
        try:
            response = requests.get(f"{url}&page={page}", headers=headers, timeout=15)
            data = response.json()
            if not data.get('data'): break
            all_data.extend(data['data'])
            if page >= data.get('last_page', 1): break
            page += 1
        except:
            break

    if all_data:
        df = pd.DataFrame(all_data)
        df['gasDayStart'] = pd.to_datetime(df['gasDayStart'])
        df['full'] = pd.to_numeric(df['full'], errors='coerce')
        df['month'] = df['gasDayStart'].dt.month

        monthly_stats = df.groupby('month')['full'].agg(['mean', 'max', 'min']).round(1)
        monthly_stats.columns = ['avg', 'max', 'min']

        stats_dict = monthly_stats.to_dict(orient='index')

        # Save averages (used for deviation)
        monthly_avg = monthly_stats['avg'].to_dict()
        with open('monthly_storage_averages.json', 'w') as f:
            json.dump(monthly_avg, f, indent=2)

        # Save full stats (used for chart)
        with open('monthly_storage_stats.json', 'w') as f:
            json.dump(stats_dict, f, indent=2)

        print(f"✅ Saved 2 JSON files (monthly averages + full stats)")
        return stats_dict
    else:
        print("No data retrieved")
        return None

# Run only once per month:
# fetch_historical_storage(years=5)

### 3.3b Current EU Storage (Run daily)

In [ ]:
# ==================== 3.3 Current EU Storage (Daily) ====================
# Pulls today's EU storage level and compares it to the historical monthly average
API_KEY = "ae22b60e0943c9239aa6027a51a140b1"

def get_current_eu_storage():
    url = "https://agsi.gie.eu/api?type=eu&size=1"
    headers = {"x-key": API_KEY}
    try:
        response = requests.get(url, headers=headers, timeout=10)
        data = response.json()
        if data.get('data'):
            latest = data['data'][0]
            return float(latest.get('full', 0)), latest.get('gasDayStart')
        return None, None
    except:
        return None, None

try:
    with open('monthly_storage_averages.json', 'r') as f:
        MONTHLY_AVG = json.load(f)
except:
    MONTHLY_AVG = {1:52, 2:45, 3:38, 4:42, 5:47, 6:55, 7:62, 8:68, 9:72, 10:75, 11:70, 12:60}

EU_STORAGE, gas_day = get_current_eu_storage()
if EU_STORAGE is None: EU_STORAGE = 33.0

current_month = datetime.now().month
monthly_avg = MONTHLY_AVG.get(str(current_month), 47)
storage_deviation = round(EU_STORAGE - monthly_avg, 1)

print(f"[3.3] EU_STORAGE = {EU_STORAGE}% | Deviation: {storage_deviation}pp")

### 3.3c Weekly Storage Data (Past 6 Months)

In [ ]:
# ==================== 3.3c Weekly Storage Data (Past 6 Months) ====================
# Pulls weekly average storage levels for Chart 1 (TTF vs Storage)
API_KEY = "ae22b60e0943c9239aa6027a51a140b1"

def get_weekly_storage_data(months=6):
    from datetime import datetime, timedelta

    end_date = datetime.now()
    start_date = end_date - timedelta(days=months*30)

    url = f"https://agsi.gie.eu/api?type=eu&from={start_date.strftime('%Y-%m-%d')}&to={end_date.strftime('%Y-%m-%d')}&size=300"
    headers = {"x-key": API_KEY}

    all_data = []
    page = 1

    while True:
        try:
            response = requests.get(f"{url}&page={page}", headers=headers, timeout=15)
            data = response.json()
            if not data.get('data'): break
            all_data.extend(data['data'])
            if page >= data.get('last_page', 1): break
            page += 1
        except:
            break

    if all_data:
        df = pd.DataFrame(all_data)
        df['gasDayStart'] = pd.to_datetime(df['gasDayStart'])
        df['full'] = pd.to_numeric(df['full'], errors='coerce')
        df.set_index('gasDayStart', inplace=True)
        weekly = df['full'].resample('W').mean().round(1)

        print(f"[3.3c] Weekly storage data fetched: {len(weekly)} weeks")
        return weekly
    else:
        print("No data retrieved")
        return None

weekly_storage = get_weekly_storage_data(months=6)

### 3.3d Current Year Monthly Storage (2026)

In [ ]:
# ==================== 3.3d Current Year Monthly Storage (2026) ====================
# Pulls monthly storage values for the current year (used for purple progress line in chart)
def get_current_year_monthly_storage():
    from datetime import datetime

    year = datetime.now().year
    start_date = f"{year}-01-01"
    end_date = datetime.now().strftime('%Y-%m-%d')

    url = f"https://agsi.gie.eu/api?type=eu&from={start_date}&to={end_date}&size=300"
    headers = {"x-key": API_KEY}

    all_data = []
    page = 1

    while True:
        try:
            response = requests.get(f"{url}&page={page}", headers=headers, timeout=15)
            data = response.json()
            if not data.get('data'): break
            all_data.extend(data['data'])
            if page >= data.get('last_page', 1): break
            page += 1
        except:
            break

    if all_data:
        df = pd.DataFrame(all_data)
        df['gasDayStart'] = pd.to_datetime(df['gasDayStart'])
        df['full'] = pd.to_numeric(df['full'], errors='coerce')
        df['month'] = df['gasDayStart'].dt.month
        monthly = df.groupby('month')['full'].last().round(1)
        print(f"[3.3d] 2026 Monthly data for months: {list(monthly.index)}")
        return monthly
    return None

current_year_data = get_current_year_monthly_storage()

### 3.4 EUA Price (Manual)

In [ ]:
# ==================== 3.4 EUA Price (Manual) ====================
# Carbon price - update manually every day
EUA_PRICE = 72.9
print(f"[3.4] EUA_PRICE = {EUA_PRICE}")

### 3.5 German Power Prices (Manual)

In [ ]:
# ==================== 3.5 German Power Prices (Manual) ====================
# Day-Ahead and 1Y Forward power prices - update manually every day
DE_DA = 62.5
DE_1Y = 91.65
print(f"[3.5] DE_DA = {DE_DA} | DE_1Y = {DE_1Y}")

### 3.6 Calculated Metrics

In [ ]:
# ==================== 3.6 Calculated Metrics ====================
# Spark spread = power price - (2 × gas price) - (0.4 × carbon price)
SPARK_SPREAD = round(DE_DA - (2.0 * TTF_PRICE) - (0.4 * EUA_PRICE), 1)
print(f"[3.6] SPARK_SPREAD = {SPARK_SPREAD}")

---
## 4. Metrics with Interpretation

In [ ]:
# ==================== 4. Metrics with Interpretation ====================
print("=== METRICS + INTERPRETATION ===\n")

print(f"1. TTF: €{TTF_PRICE} | 30d Change: {ttf_data['change_30d']}% | Range: €{ttf_data['low_30d']}-€{ttf_data['high_30d']}")
print(f"2. TTF 30d Realized Volatility = {TTF_VOL}%")
print(f"3. Storage: {EU_STORAGE}% ({storage_deviation}pp vs monthly avg)")
print(f"4. EUA: €{EUA_PRICE}")
print(f"5. German Day-Ahead: €{DE_DA}")
print(f"6. German 1Y Forward: €{DE_1Y}")
print(f"7. Spark Spread: €{SPARK_SPREAD}")

---
## 5. Charts

### 5.1 Chart 1: EU Gas Storage Development vs 5-Year Average

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime

# Load 5-year stats
with open('monthly_storage_stats.json', 'r') as f:
    stats = json.load(f)

df_5y = pd.DataFrame.from_dict(stats, orient='index')
df_5y.index = df_5y.index.astype(int)
df_5y = df_5y.sort_index()

current_year = datetime.now().year
current_month = datetime.now().month

month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
               'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

plt.figure(figsize=(13, 6))
plt.fill_between(df_5y.index, df_5y['min'], df_5y['max'],
                 color='#9b59b6', alpha=0.15, label='5-Year Min/Max')
plt.plot(df_5y.index, df_5y['avg'], color='black', linewidth=2.5, label='5-Year Average')

if current_year_data is not None:
    plt.plot(current_year_data.index, current_year_data.values,
             color='#6a0dad', linewidth=3, label=f'{current_year} Progress')
    plt.scatter(current_year_data.index, current_year_data.values,
                color='#6a0dad', s=80, zorder=5)

plt.title('Development of EU Gas Storage vs 5-Year Average', fontsize=14, fontweight='bold')
plt.xlabel('Month')
plt.ylabel('Storage Level (%)')
plt.xticks(df_5y.index, month_names)
plt.legend(loc='upper right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('chart_1.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close()

### 5.2 Chart 2: TTF Price vs EU Storage (Real 6-Month Data)

In [ ]:
fig1, ax1 = plt.subplots(figsize=(11, 5))

ttf_6m_naive = ttf_6m.tz_localize(None)
combined = pd.concat([ttf_6m_naive.resample('W').mean(), weekly_storage], axis=1)
combined.columns = ['TTF', 'Storage']
combined = combined.dropna()

dates = combined.index
ttf_p = combined['TTF'].values.round(2)
storage_p = combined['Storage'].values

ax1.plot(dates, ttf_p, color='#1f77b4', linewidth=2.5, label='TTF (Weekly Avg)')
last_ttf_weekly = round(ttf_p[-1], 2)
ax1.axhline(last_ttf_weekly, color='#1f77b4', linestyle='--', alpha=0.7,
            label=f'Latest Weekly: €{last_ttf_weekly} ({ttf_data["change_30d"]}% 30d)')
ax1.set_ylabel('TTF (EUR/MWh)', color='#1f77b4', fontsize=11)

ax2 = ax1.twinx()
ax2.plot(dates, storage_p, color='#ff7f0e', linewidth=2.5, linestyle='--', label='EU Storage (Weekly Avg)')
ax2.axhline(EU_STORAGE, color='#ff7f0e', linestyle=':', alpha=0.8,
            label=f'Current: {EU_STORAGE}% ({storage_deviation}pp)')
ax2.set_ylabel('Storage Level (%)', color='#ff7f0e', fontsize=11)

plt.title('TTF Price vs EU Storage (Real 6-Month Data)', fontsize=12, fontweight='bold')
ax1.legend(loc='upper left')
ax2.legend(loc='upper right')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('chart_2.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close()

### 5.3 Chart 3: EUA vs German Power Forward Curve (Simulated)

In [ ]:
fig2, ax = plt.subplots(figsize=(11, 5))
months = pd.date_range('2025-05-01', '2026-05-01', freq='ME')
eua_p = np.array([68,71,74,72,69,75,78,76,73,71,74,73,72.9])[:len(months)]
power_p = np.array([82,85,88,90,92,95,98,96,93,91,94,92,91.65])[:len(months)]
ax.plot(months, eua_p, 'g-o', linewidth=2.5, label='EUA')
ax.plot(months, power_p, 'r-s', linewidth=2.5, label='Germany 1Y Forward')
ax.axhline(EUA_PRICE, color='green', linestyle='--', alpha=0.6)
ax.axhline(DE_1Y, color='red', linestyle='--', alpha=0.6)
plt.title('EUA vs German Power Forward', fontsize=12, fontweight='bold')
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('chart_3.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close()

### 5.4 Chart 4: Price & Storage Dashboard

In [ ]:
fig3, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

EUA_PER_MWH = round(EUA_PRICE * 0.4, 1)
GENERATION_COST = round((2.0 * TTF_PRICE) + (0.4 * EUA_PRICE), 1)

names = ['TTF', 'EUA Cost/MWh', 'Generation Cost', 'DE DA', 'DE 1Y']
vals = [TTF_PRICE, EUA_PER_MWH, GENERATION_COST, DE_DA, DE_1Y]

bars = ax1.bar(names, vals, color=['#1f77b4','#2ca02c','#8c564b','#d62728','#9467bd'], edgecolor='black')
ax1.set_title('Price Snapshot', fontweight='bold')
for b, v in zip(bars, vals):
    ax1.text(b.get_x() + b.get_width()/2, b.get_height()+1, f'{v:.1f}', ha='center', fontweight='bold')

ax2.barh(['EU Storage'], [EU_STORAGE], color='#ff7f0e', edgecolor='black')
ax2.axvline(x=monthly_avg, color='red', linestyle='--', linewidth=2,
            label=f'Monthly Avg ({current_month}): {monthly_avg}%')
ax2.set_title('Storage Level', fontweight='bold')
ax2.legend()

plt.suptitle('Price and Storage Dashboard', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('chart_4.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close()

### 5.5 Chart 5 TTF Price vs 30-Day Realized Volatility (Real Data)

In [ ]:
fig4, ax1 = plt.subplots(figsize=(11, 5))

combined = pd.concat([ttf_6m.resample('W').mean(), historical_vol], axis=1)
combined.columns = ['TTF', 'Volatility']
combined = combined.dropna()

dates = combined.index
ttf_p = combined['TTF'].values.round(2)
vol_p = combined['Volatility'].values

ax1.plot(dates, ttf_p, color='#1f77b4', linewidth=2.5, label='TTF Price (Weekly Avg)')
ax1.set_ylabel('TTF (EUR/MWh)', color='#1f77b4', fontsize=11)

ax2 = ax1.twinx()
ax2.plot(dates, vol_p, color='#d62728', linewidth=2.5, linestyle='--', label='30d Realized Vol')
ax2.axhline(TTF_VOL, color='#d62728', linestyle=':', alpha=0.8, label=f'Current: {TTF_VOL}%')
ax2.set_ylabel('Volatility (%)', color='#d62728', fontsize=11)

plt.title('TTF Price vs 30-Day Realized Volatility (Real Data)', fontsize=12, fontweight='bold')
ax1.legend(loc='upper left')
ax2.legend(loc='upper right')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('chart_5.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close()

---
## 6. Daily Summary

In [ ]:
# ==================== 6. Daily Cross-Commodity Risk Summary ====================
lines = []

def log(text=""):
    lines.append(text)
    print(text)

log("=== DAILY CROSS-COMMODITY RISK SUMMARY ===\n")

log("Gas Market:")
log(f"  • TTF: €{TTF_PRICE} ({ttf_data['change_30d']}% 30d) | Range: €{ttf_data['low_30d']}–€{ttf_data['high_30d']} | Avg: €{ttf_data['avg_30d']}")
log(f"  • 30d Realized Vol: {TTF_VOL}%")

if TTF_PRICE > 55:   ttf_signal = RULES['TTF']['tight'];    ttf_state = "Tight"
elif TTF_PRICE < 40: ttf_signal = RULES['TTF']['loose'];    ttf_state = "Loose"
else:                ttf_signal = RULES['TTF']['moderate']; ttf_state = "Moderate"
log(f"  • TTF Signal: {ttf_signal}")

if storage_deviation < -10:  storage_signal = STORAGE_RULES['critical'];    storage_state = "Critical"
elif storage_deviation < -5: storage_signal = STORAGE_RULES['tight'];       storage_state = "Tight"
elif storage_deviation > 5:  storage_signal = STORAGE_RULES['comfortable']; storage_state = "Comfortable"
else:                        storage_signal = STORAGE_RULES['normal'];      storage_state = "Normal"
log(f"  • EU Storage: {EU_STORAGE}% ({storage_deviation}pp vs monthly avg) → {storage_signal}")

log("\nCarbon & Power:")
log(f"  • EUA: €{EUA_PRICE}/tCO₂ (Carbon cost/MWh: €{round(EUA_PRICE*0.4,1)})")
if EUA_PRICE > 75:   eua_signal = RULES['EUA']['strong'];       eua_state = "Strong"
elif EUA_PRICE < 60: eua_signal = RULES['EUA']['weak'];         eua_state = "Weak"
else:                eua_signal = RULES['EUA']['constructive']; eua_state = "Normal"
log(f"  • EUA Signal: {eua_signal}")
log(f"  • German Power: DA €{DE_DA} | 1Y Forward €{DE_1Y} (Premium: €{round(DE_1Y - DE_DA, 2)})")

spark_signal = RULES['Spark_Spread']['positive'] if SPARK_SPREAD > 0 else RULES['Spark_Spread']['negative']
log(f"  • Spark Spread: €{SPARK_SPREAD}/MWh → {spark_signal}")

log("\nRisk Assessment:")
risk_score = 0
if TTF_PRICE > 55:         risk_score += 2
elif TTF_PRICE < 40:       risk_score -= 1

if TTF_VOL > 50:           risk_score += 3; vol_state = "Very High"
elif TTF_VOL > 35:         risk_score += 1; vol_state = "Elevated"
else:                                        vol_state = "Low"

if storage_deviation < -10:   risk_score += 3
elif storage_deviation < -5:  risk_score += 2
elif storage_deviation > 5:   risk_score -= 1

if EUA_PRICE > 75:         risk_score += 1

if risk_score >= 7:   risk_level = "CRITICAL"; risk_text = "Explosive upside risk for power curve"
elif risk_score >= 5: risk_level = "HIGH";     risk_text = "Strong bullish bias for winter 2026/27 power curve"
elif risk_score >= 2: risk_level = "ELEVATED"; risk_text = "Bullish bias for power curve"
elif risk_score >= 0: risk_level = "NEUTRAL";  risk_text = "Balanced conditions — monitor closely"
else:                 risk_level = "LOW";      risk_text = "Bearish bias for power curve"

if risk_level == "CRITICAL":  rec = "Strong bullish signal — be long power curve but buy protective calls"
elif risk_level == "HIGH":    rec = "Bullish bias — consider long power curve positions"
elif risk_level == "ELEVATED":rec = "Mildly bullish — small long bias or stay neutral"
elif risk_level == "NEUTRAL": rec = "No clear direction — stay neutral or range trade"
else:                         rec = "Bearish bias — consider short power exposure or stay neutral"

log(f"  • Risk Level: {risk_level} (Score: {risk_score}/10)")
log(f"    Components → Gas: {ttf_state} | Vol: {vol_state} | Storage: {storage_state} | EUA: {eua_state}")
log(f"  • {risk_text}")
log(f"  • {rec}")

RISK_SUMMARY_TEXT = "\n".join(lines)
print("\n[Summary captured for PDF export]")

## LLM reasoning and daily brief PDF generation

In [ ]:
# ==================== 7.1 Ollama Desk Note Generation ====================
import requests as _req

def generate_desk_note(summary_text):
    prompt = f"""You are a senior European energy market analyst. Based on the cross-commodity risk data below, write a concise analyst desk note.

MARKET DATA:
{summary_text}

Write the desk note in EXACTLY this format with these section headers (no extra text before or after):

MARKET OVERVIEW
[2-3 sentences on current European energy market conditions based on the data]

KEY DRIVERS
• [TTF gas price driver and what it signals]
• [EU storage deviation driver and urgency]
• [EUA carbon price driver and power cost implication]
• [Spark spread and what it means for gas generation]

RISK ASSESSMENT
[One paragraph: risk score meaning, what components are elevated, implication for winter 2026/27 power curve]

TRADING RECOMMENDATION
[1-2 sentences: directional call on winter power curve with conviction level]

RISKS TO VIEW
• [Key upside risk if bullish, or downside risk if bearish]
• [Second risk factor that could change the view]"""

    try:
        resp = _req.post(
            'http://localhost:11434/api/generate',
            json={'model': 'llama3.1', 'prompt': prompt, 'stream': False},
            timeout=120
        )
        resp.raise_for_status()
        desk_note = resp.json()['response'].strip()
        print("✅ Desk note generated by llama3.1\n")
        print(desk_note)
        return desk_note
    except Exception as e:
        print(f"❌ Ollama error: {e}")
        return None

DESK_NOTE = generate_desk_note(RISK_SUMMARY_TEXT)

In [ ]:
# ==================== 7.2 PDF Export ====================
import os
from fpdf import FPDF
from datetime import datetime

CHART_DIR = os.getcwd()
print(f"[PDF] Chart dir: {CHART_DIR}")
print(f"[PDF] PNGs found: {[f for f in os.listdir(CHART_DIR) if f.endswith('.png')]}")

RISK_COLORS = {
    'CRITICAL': (180, 0,   0),
    'HIGH':     (210, 60,  0),
    'ELEVATED': (200, 140, 0),
    'NEUTRAL':  (100, 100, 100),
    'LOW':      (0,   140, 0),
}

CHART_TITLES = [
    'Chart 1: EU Gas Storage vs 5-Year Average',
    'Chart 2: TTF Price vs EU Storage (6-Month)',
    'Chart 3: EUA vs German Power Forward',
    'Chart 4: Price & Storage Dashboard',
    'Chart 5: TTF Price vs 30-Day Realized Volatility',
]

def sanitize(text):
    replacements = {
        '€': 'EUR', '→': '->', '•': '-', '–': '-',
        '—': '--', '‘': "'", '’': "'",
        '“': '"', '”': '"', '°': 'deg', '²': '2',
    }
    for char, replacement in replacements.items():
        text = text.replace(char, replacement)
    return text.encode('latin-1', errors='replace').decode('latin-1')

def build_pdf(desk_note_text):
    timestamp = datetime.now().strftime('%Y%m%d_%H%M')
    output_path = os.path.join(CHART_DIR, f'EU_Risk_Monitor_{timestamp}.pdf')

    pdf = FPDF()
    pdf.set_auto_page_break(auto=True, margin=15)
    pdf.set_margins(15, 15, 15)

    # ── PAGE 1 ──────────────────────────────────────────────────────────────
    pdf.add_page()

    pdf.set_fill_color(20, 40, 80)
    pdf.set_text_color(255, 255, 255)
    pdf.set_font('Helvetica', 'B', 15)
    pdf.cell(0, 13, 'European Cross-Commodity Risk Monitor', fill=True, align='C', ln=1)
    pdf.set_font('Helvetica', '', 9)
    pdf.set_fill_color(50, 70, 120)
    pdf.cell(0, 7, f'{datetime.now().strftime("%d %B %Y")}  |  v12.0  |  Gas + Carbon -> Power Curve  |  Author: Clifford Pang',
             fill=True, align='C', ln=1)
    pdf.set_text_color(0, 0, 0)
    pdf.ln(6)

    # ── SECTION 1: Metrics Snapshot ─────────────────────────────────────────
    pdf.set_font('Helvetica', 'B', 11)
    pdf.set_fill_color(220, 228, 245)
    pdf.cell(0, 8, '  1. Metrics Snapshot', fill=True, ln=1)
    pdf.ln(3)

    col_w = [58, 42, 52, 28]
    pdf.set_font('Helvetica', 'B', 9)
    pdf.set_fill_color(190, 205, 230)
    for header, w in zip(['Metric', 'Value', 'Signal', 'Change'], col_w):
        pdf.cell(w, 7, header, border=1, fill=True, align='C')
    pdf.ln()

    ttf_signal_label  = {'Tight': '>55: Tight', 'Moderate': '45-55: Moderate', 'Loose': '<40: Loose'}.get(ttf_state, ttf_state)
    vol_signal_label  = {'Very High': 'Very High (>50%)', 'Elevated': 'Elevated (>35%)', 'Low': 'Low (<35%)'}.get(vol_state, vol_state)
    stor_signal_label = {'Critical': f'Critical ({storage_deviation}pp)', 'Tight': f'Tight ({storage_deviation}pp)',
                         'Normal': f'Normal ({storage_deviation}pp)', 'Comfortable': f'Comfortable ({storage_deviation}pp)'}.get(storage_state, storage_state)
    eua_signal_label  = {'Strong': '>75: Strong', 'Normal': '65-75: Constructive', 'Weak': '<60: Weak'}.get(eua_state, eua_state)
    spark_status      = 'Gas unprofitable' if SPARK_SPREAD < 0 else 'Gas profitable'
    spark_signal_label = 'NEGATIVE -> Bullish' if SPARK_SPREAD < 0 else 'POSITIVE -> Bearish'

    rows = [
        ['TTF Gas Price',     f'EUR{TTF_PRICE}/MWh',   ttf_signal_label,   f'{ttf_data["change_30d"]}% (30d)'],
        ['TTF Realized Vol',  f'{TTF_VOL}%',            vol_signal_label,   '30d annualised'],
        ['EU Storage',        f'{EU_STORAGE}%',         stor_signal_label,  f'vs {monthly_avg}% avg'],
        ['EUA Carbon',        f'EUR{EUA_PRICE}/tCO2',  eua_signal_label,   f'Cost/MWh: EUR{round(EUA_PRICE*0.4,1)}'],
        ['German DA Power',   f'EUR{DE_DA}/MWh',        '-',                '-'],
        ['German 1Y Forward', f'EUR{DE_1Y}/MWh',        '-',                f'Prem: EUR{round(DE_1Y-DE_DA,1)}'],
        ['Spark Spread',      f'EUR{SPARK_SPREAD}/MWh', spark_signal_label, spark_status],
    ]

    pdf.set_font('Helvetica', '', 8)
    for i, row in enumerate(rows):
        pdf.set_fill_color(245, 248, 252) if i % 2 == 0 else pdf.set_fill_color(255, 255, 255)
        for cell_text, w in zip(row, col_w):
            pdf.cell(w, 6, cell_text, border=1, fill=True, align='C')
        pdf.ln()
    pdf.ln(5)

    # ── SECTION 2: Risk Score ────────────────────────────────────────────────
    pdf.set_font('Helvetica', 'B', 11)
    pdf.set_fill_color(220, 228, 245)
    pdf.cell(0, 8, '  2. Risk Assessment', fill=True, ln=1)
    pdf.ln(3)

    r, g, b = RISK_COLORS.get(risk_level, (100, 100, 100))
    pdf.set_fill_color(r, g, b)
    pdf.set_text_color(255, 255, 255)
    pdf.set_font('Helvetica', 'B', 13)
    pdf.cell(0, 11, f'RISK LEVEL: {risk_level}   |   Score: {risk_score} / 10', fill=True, align='C', ln=1)
    pdf.set_text_color(0, 0, 0)
    pdf.set_font('Helvetica', '', 9)
    pdf.set_fill_color(240, 242, 248)
    pdf.cell(0, 7, f'Gas: {ttf_state}   |   Volatility: {vol_state}   |   Storage: {storage_state}   |   EUA: {eua_state}',
             border=1, fill=True, align='C', ln=1)
    pdf.ln(5)

    # ── SECTION 3: Desk Note ─────────────────────────────────────────────────
    pdf.set_font('Helvetica', 'B', 11)
    pdf.set_fill_color(220, 228, 245)
    pdf.cell(0, 8, '  3. Analyst Desk Note  (generated by llama3.1)', fill=True, ln=1)
    pdf.ln(3)

    SECTION_HEADERS = {'MARKET OVERVIEW', 'KEY DRIVERS', 'RISK ASSESSMENT',
                       'TRADING RECOMMENDATION', 'RISKS TO VIEW'}
    text_width = pdf.w - pdf.l_margin - pdf.r_margin

    if desk_note_text:
        for line in desk_note_text.split('\n'):
            line = sanitize(line.strip())
            if not line:
                pdf.ln(2)
                continue
            if line in SECTION_HEADERS:
                pdf.set_font('Helvetica', 'B', 10)
                pdf.set_text_color(20, 40, 80)
                pdf.set_x(pdf.l_margin)
                pdf.cell(0, 6, line, ln=1)
                pdf.set_font('Helvetica', '', 9)
                pdf.set_text_color(0, 0, 0)
            else:
                pdf.set_x(pdf.l_margin)
                pdf.multi_cell(text_width, 5, line)
    else:
        pdf.set_font('Helvetica', 'I', 9)
        pdf.set_x(pdf.l_margin)
        pdf.cell(0, 6, 'Desk note unavailable (Ollama not reachable).', ln=1)

    # ── PAGE 2: Charts 1–3 ───────────────────────────────────────────────────
    pdf.add_page()
    pdf.set_font('Helvetica', 'B', 11)
    pdf.set_fill_color(220, 228, 245)
    pdf.cell(0, 8, '  4. Charts', fill=True, ln=1)
    pdf.ln(3)

    for i in range(1, 4):
        chart_path = os.path.join(CHART_DIR, f'chart_{i}.png')
        if os.path.exists(chart_path):
            pdf.set_font('Helvetica', 'B', 9)
            pdf.set_text_color(60, 60, 60)
            pdf.set_x(pdf.l_margin)
            pdf.cell(0, 5, CHART_TITLES[i - 1], ln=1)
            pdf.set_text_color(0, 0, 0)
            pdf.image(chart_path, x=25, w=160)
            pdf.ln(3)
        else:
            pdf.set_font('Helvetica', 'I', 8)
            pdf.set_x(pdf.l_margin)
            pdf.cell(0, 5, f'{CHART_TITLES[i - 1]} - not found', ln=1)
            pdf.ln(2)

    # ── PAGE 3: Charts 4–5 ───────────────────────────────────────────────────
    pdf.add_page()
    pdf.set_font('Helvetica', 'B', 11)
    pdf.set_fill_color(220, 228, 245)
    pdf.cell(0, 8, '  4. Charts (continued)', fill=True, ln=1)
    pdf.ln(3)

    for i in range(4, 6):
        chart_path = os.path.join(CHART_DIR, f'chart_{i}.png')
        if os.path.exists(chart_path):
            pdf.set_font('Helvetica', 'B', 9)
            pdf.set_text_color(60, 60, 60)
            pdf.set_x(pdf.l_margin)
            pdf.cell(0, 5, CHART_TITLES[i - 1], ln=1)
            pdf.set_text_color(0, 0, 0)
            pdf.image(chart_path, x=15, w=180)
            pdf.ln(5)
        else:
            pdf.set_font('Helvetica', 'I', 8)
            pdf.set_x(pdf.l_margin)
            pdf.cell(0, 5, f'{CHART_TITLES[i - 1]} - not found', ln=1)
            pdf.ln(2)

    # Footer — disable auto page break so set_y(-12) doesn't spill onto a new page
    pdf.set_auto_page_break(False)
    pdf.set_y(-12)
    pdf.set_font('Helvetica', 'I', 7)
    pdf.set_text_color(150, 150, 150)
    pdf.cell(0, 5, f'Generated {datetime.now().strftime("%d %b %Y %H:%M")} | EU Cross-Commodity Risk Monitor v12 | Powered by llama3.1', align='C')

    pdf.output(output_path)
    print(f"\n✅ PDF saved: {output_path}")
    return output_path

build_pdf(DESK_NOTE)